# YOLO11n — Classification Output Layer Only (BDD100K)

**Experiment:** Adapt only the three final class-output convolutions of YOLO11n to BDD100K. All backbone, neck, classification feature layers, box-regression layers, and DFL remain frozen.

This is a small preliminary run using 5% of the training split; it is not a final benchmark.

In [1]:
from pathlib import Path
import torch
import pandas as pd
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = 0 if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Training device:", DEVICE)

PyTorch: 2.7.1+cu128
CUDA available: True
GPU: NVIDIA GeForce GTX 1650
Training device: 0


In [2]:
# Paths
PROJECT_DIR = Path.cwd().parent  
DATA_YAML = PROJECT_DIR / "dataset" / "processed" / "data.yaml"
OUTPUT_DIR = PROJECT_DIR / "outputs"
RUN_NAME = "yolo11n_bdd100k_classification"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_YAML.exists(), f"Dataset YAML not found: {DATA_YAML}"
print("Dataset:", DATA_YAML)
LAST_CHECKPOINT = OUTPUT_DIR / RUN_NAME / "weights" / "last.pt"
print("Run output:", OUTPUT_DIR / RUN_NAME)
print("Resume checkpoint:", LAST_CHECKPOINT)

Dataset: Y:\Python\AI ML\Month 2\week 8-NLP\50 — TESLA AUTOPILOT CLONE — OBJECT DETECTION\dataset\processed\data.yaml
Run output: Y:\Python\AI ML\Month 2\week 8-NLP\50 — TESLA AUTOPILOT CLONE — OBJECT DETECTION\outputs\yolo11n_bdd100k_classification
Resume checkpoint: Y:\Python\AI ML\Month 2\week 8-NLP\50 — TESLA AUTOPILOT CLONE — OBJECT DETECTION\outputs\yolo11n_bdd100k_classification\weights\last.pt


## 1. Load fresh pretrained model or resume automatically

If `outputs/RUN_NAME/weights/last.pt` exists, the notebook loads it and enables resume mode.
Otherwise, it loads a fresh COCO-pretrained `yolo11n.pt` model.


In [3]:
# Automatically resume from the last saved checkpoint if it exists.
# Otherwise, start from the fresh COCO-pretrained YOLO11n model.
RESUME_MODE = LAST_CHECKPOINT.exists()

if RESUME_MODE:
    model = YOLO(str(LAST_CHECKPOINT))
    print(f"Found checkpoint. Resume mode enabled: {LAST_CHECKPOINT}")
else:
    model = YOLO("yolo11n.pt")
    print("No checkpoint found. Loaded fresh COCO-pretrained YOLO11n.")


Found checkpoint. Resume mode enabled: Y:\Python\AI ML\Month 2\week 8-NLP\50 — TESLA AUTOPILOT CLONE — OBJECT DETECTION\outputs\yolo11n_bdd100k_classification\weights\last.pt


## 2. Freeze everything, then unfreeze only the final class-output convolutions

The model architecture has a Detect head at index 23. In each of its three `cv3` scales, module `[2]` is the final class-output convolution. Only those three modules are trainable.

In [4]:
import torch
import torch.nn as nn

detect_head = model.model.model[23]

# Adapt COCO 80-class output layers only when starting a new run.
# A saved checkpoint already contains the 10-class BDD100K head.
if not RESUME_MODE:
    for scale in range(3):
        old_layer = detect_head.cv3[scale][2]

        new_layer = nn.Conv2d(
            in_channels=old_layer.in_channels,
            out_channels=10,
            kernel_size=old_layer.kernel_size,
            stride=old_layer.stride,
            padding=old_layer.padding,
            bias=True
        ).to(device=old_layer.weight.device, dtype=old_layer.weight.dtype)

        detect_head.cv3[scale][2] = new_layer
else:
    print("Resume mode: retaining the checkpoint's existing 10-class output layers.")

# Freeze everything
for param in model.model.parameters():
    param.requires_grad = False

# Train only the three new classification output convolutions
for scale in range(3):
    for param in detect_head.cv3[scale][2].parameters():
        param.requires_grad = True

Resume mode: retaining the checkpoint's existing 10-class output layers.


## 3. Parameter-count verification

Expected for the inspected 10-class head: **2,430 trainable parameters**. If the count differs, stop and inspect the head instead of training.

In [5]:
for name, param in model.model.named_parameters():
    if param.requires_grad:
        print(f"{name:70} {param.numel():>8,}")

model.23.cv3.0.2.weight                                                     800
model.23.cv3.0.2.bias                                                        10
model.23.cv3.1.2.weight                                                     800
model.23.cv3.1.2.bias                                                        10
model.23.cv3.2.2.weight                                                     800
model.23.cv3.2.2.bias                                                        10


In [6]:
total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters   : {frozen_params:,}")
print(f"Trainable %        : {trainable_params / total_params * 100:.4f}%")
assert trainable_params == 2430, f"Expected 2,430 trainable params, got {trainable_params:,}"

Total parameters     : 2,607,070
Trainable parameters: 2,430
Frozen parameters   : 2,604,640
Trainable %        : 0.0932%


## 4. Training configuration

`freeze=0` is intentional: the custom trainer handles selective freezing.
This configuration uses 20% of the training set for 15 epochs and disables validation during training.
The output directory is stable (`exist_ok=True`) so the checkpoint path is predictable for resuming.


In [7]:
IMAGE_SIZE = 512
BATCH_SIZE = 4
WORKERS = 0

TRAIN_ARGS = {
    "data": str(DATA_YAML),
    "epochs": 15,
    "imgsz": IMAGE_SIZE,
    "batch": BATCH_SIZE,
    "workers": WORKERS,
    "device": DEVICE,
    "freeze": 0,
    "fraction": 0.20,
    "optimizer": "auto",
    "weight_decay": 0.0005,
    "amp": False,             
    "patience": 5,
    "cache": False,
    "save": True,
    "save_period": -1,
    "val": False,
    "plots": False,          
    "project": str(OUTPUT_DIR),
    "name": RUN_NAME,
    "exist_ok": True,
    "seed": 42,
    "deterministic": True,
    "verbose": True,
}

pd.Series(TRAIN_ARGS)

data             Y:\Python\AI ML\Month 2\week 8-NLP\50 — TESLA ...
epochs                                                          15
imgsz                                                          512
batch                                                            4
workers                                                          0
device                                                           0
freeze                                                           0
fraction                                                       0.2
optimizer                                                     auto
weight_decay                                                0.0005
amp                                                          False
patience                                                         5
cache                                                        False
save                                                          True
save_period                                                   

## 5. Trainable-set safety check

The custom trainer checks that exactly **2,430 parameters** are trainable after its setup.
If the trainable count is different, it aborts training.


In [8]:
# Trainable-set validation is performed inside ClassificationOnlyTrainer.
# Avoid registering callbacks on a model that may be reloaded from last.pt.
print("Trainable-set safety check is built into the custom trainer.")


Trainable-set safety check is built into the custom trainer.


## 6. Resumable training

This cell resumes from `last.pt` when the checkpoint exists. Otherwise, it starts a new run.
Ultralytics resumes optimizer, scheduler, scaler, and epoch state from the checkpoint.


In [9]:
from ultralytics.models.yolo.detect import DetectionTrainer

class ClassificationOnlyTrainer(DetectionTrainer):

    def _setup_train(self):
        # Let Ultralytics initialize training normally
        super()._setup_train()

        # Freeze everything again
        for param in self.model.parameters():
            param.requires_grad = False

        # Access the Detect head
        detect_head = self.model.model[23]

        # Unfreeze ONLY the 3 final 10-class output convolutions
        for scale in range(3):
            for param in detect_head.cv3[scale][2].parameters():
                param.requires_grad = True

        # Verify trainable count
        trainable_params = sum(
            p.numel() for p in self.model.parameters() if p.requires_grad
        )

        print(f"\n[CLASSIFICATION-ONLY CHECK] Trainable parameters: {trainable_params:,}")

        if trainable_params != 2430:
            raise RuntimeError(
                f"Expected 2,430 trainable parameters, got {trainable_params:,}. "
                "Training aborted."
            )

In [10]:
if RESUME_MODE:
    checkpoint = torch.load(LAST_CHECKPOINT, map_location="cpu", weights_only=False)
    saved_args = checkpoint.get("train_args", checkpoint.get("args", {}))
    saved_epochs = saved_args.get("epochs", TRAIN_ARGS["epochs"])
    next_epoch = checkpoint.get("epoch", -1) + 1

    if next_epoch >= saved_epochs:
        raise RuntimeError(
            f"This checkpoint is already complete ({next_epoch}/{saved_epochs} epochs). "
            "To start a new experiment, change RUN_NAME in Cell 2 to a new folder name."
        )

    print(f"Resuming from checkpoint: {LAST_CHECKPOINT}")
    print(f"Completed epochs: {next_epoch}/{saved_epochs}")
    print("Resume restores the checkpoint's saved training settings.")
    results = model.train(
        resume=True,
        trainer=ClassificationOnlyTrainer,
        **TRAIN_ARGS
    )
else:
    print("Starting a new run from COCO pretrained weights.")
    results = model.train(
        trainer=ClassificationOnlyTrainer,
        **TRAIN_ARGS
    )


Resuming from checkpoint: Y:\Python\AI ML\Month 2\week 8-NLP\50 — TESLA AUTOPILOT CLONE — OBJECT DETECTION\outputs\yolo11n_bdd100k_classification\weights\last.pt
Completed epochs: 6/15
Resume restores the checkpoint's saved training settings.
New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
WARNING Resume ignores ['optimizer', 'amp'], using checkpoint values. Start a new run to change them.
Ultralytics 8.4.194  Python-3.10.11 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Y:\Python\AI ML\Month 2\week 8-NLP\50 — TESLA AUTOPILOT CLONE — OBJECT DETECTION\dataset\processed\data.yaml, degrees=0.0, dete

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



      13/15     0.199G        1.9      1.589      1.225         32        512: 100% ━━━━━━━━━━━━ 3494/3494 7.5it/s 7:460.3ss

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      14/15     0.203G      1.902      1.587      1.225          7        512: 100% ━━━━━━━━━━━━ 3494/3494 7.6it/s 7:380.1ss

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      15/15     0.211G      1.902      1.586      1.224         20        512: 100% ━━━━━━━━━━━━ 3494/3494 7.9it/s 7:250.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1250/1250 5.7it/s 3:380.2ss
                   all      10000     185526      0.343      0.134      0.101     0.0522

9 epochs completed in 1.196 hours.
Optimizer stripped from Y:\Python\AI ML\Month 2\week 8-NLP\50 — TESLA AUTOPILOT CLONE — OBJECT DETECTION\outputs\yolo11n_bdd100k_classification\weights\last.pt, 5.5MB
Optimizer stripped from Y:\Python\AI ML

## 7. Post-training checkpoint paths and parameter check

In [11]:
weights_dir = OUTPUT_DIR / RUN_NAME / "weights"
BEST_CHECKPOINT = weights_dir / "best.pt"
LAST_CHECKPOINT = weights_dir / "last.pt"

print("Best checkpoint exists:", BEST_CHECKPOINT.exists(), BEST_CHECKPOINT)
print("Last checkpoint exists:", LAST_CHECKPOINT.exists(), LAST_CHECKPOINT)


Best checkpoint exists: True Y:\Python\AI ML\Month 2\week 8-NLP\50 — TESLA AUTOPILOT CLONE — OBJECT DETECTION\outputs\yolo11n_bdd100k_classification\weights\best.pt
Last checkpoint exists: True Y:\Python\AI ML\Month 2\week 8-NLP\50 — TESLA AUTOPILOT CLONE — OBJECT DETECTION\outputs\yolo11n_bdd100k_classification\weights\last.pt
